# 030 · Weight Initialization, Part 2 — Xavier/Glorot and He

Lesson 029 showed that a fixed scale is always wrong for *some* layer. The fix
is to make the spread depend on the layer's **fan-in**.

| Part | What we reproduce |
|---|---|
| A | Var(z) = fan_in × Var(w) × Var(x), measured |
| B | the 12-layer comparison: **He 0.83 → 0.98**, Xavier 0.63 → 0.21, ×0.01 dead by layer 6 |
| C | where the factor of **2** in He comes from — derived, not tuned |
| D | Keras defaults to `glorot_uniform` even on ReLU layers |

Needs `numpy`. Part D needs `tensorflow`, and is written to be skipped.

In [ ]:
import numpy as np

N, WIDTH, DEPTH = 512, 256, 12
tanh = np.tanh
def relu(z):
    return np.maximum(0.0, z)

## Part A — Why fan-in is the right thing to divide by

A neuron computes `z = sum over fan_in of w * x`. If the terms are independent
and zero-mean, their variances add:

$$\text{Var}(z) = \text{fan\_in} \times \text{Var}(w) \times \text{Var}(x)$$

So to keep `Var(z) = Var(x)` — the signal neither growing nor shrinking — you
want `Var(w) = 1 / fan_in`. Check that this is really true rather than taking it
on trust.

In [ ]:
rng = np.random.default_rng(0)

print(f"{'fan_in':>8}{'Var(w)':>12}{'predicted':>12}{'measured':>12}")
for fan_in in (5, 50, 500, 5000):
    x = rng.standard_normal((4000, fan_in))     # Var(x) = 1
    var_w = 0.37                                 # any value; the law is general
    W = rng.standard_normal((fan_in, 1)) * np.sqrt(var_w)
    z = x @ W
    predicted = fan_in * var_w * 1.0
    print(f"{fan_in:>8}{var_w:>12}{predicted:>12.1f}{z.var():>12.1f}")

print("\nThe prediction tracks the measurement. So Var(w) = 1/fan_in is the")
print("choice that leaves Var(z) equal to Var(x), whatever the layer width.")

In [ ]:
# And it self-adjusts. You never pick a number - the layer picks it.
for fan_in in (16, 256, 4096):
    print(f"  fan_in {fan_in:>5} -> std of weights = {(1/fan_in) ** 0.5:.4f}")
print("\nWide layer -> smaller weights. Narrow layer -> larger. Automatically.")

## Part B — The 12-layer comparison

Now run all the schemes side by side through 12 layers and watch which ones hold
their signal.

- **Xavier / Glorot:** `Var(w) = 2 / (fan_in + fan_out)` — balances the forward
  and backward passes. **For tanh and sigmoid.**
- **He:** `Var(w) = 2 / fan_in`. **For ReLU and its variants.**

In [ ]:
def propagate(scale, act, seed=0):
    rng = np.random.default_rng(seed)
    a = rng.standard_normal((N, WIDTH))
    stds = []
    for _ in range(DEPTH):
        W = rng.standard_normal((a.shape[1], WIDTH)) * scale(a.shape[1])
        a = act(a @ W)
        stds.append(float(a.std()))
    return stds


SCHEMES = [
    ("small x0.01  (tanh)", lambda fan: 0.01, tanh),
    ("large x1.0   (tanh)", lambda fan: 1.0, tanh),
    ("Xavier 1/sqrt(fan)", lambda fan: (1 / fan) ** 0.5, tanh),
    ("small x0.01  (ReLU)", lambda fan: 0.01, relu),
    ("He 2/sqrt(fan) (ReLU)", lambda fan: (2 / fan) ** 0.5, relu),
]

print(f"{'scheme':<24}" + "".join(f"L{i+1:<7}" for i in (0, 2, 5, 8, 11)))
for name, scale, act in SCHEMES:
    stds = propagate(scale, act)
    print(f"{name:<24}" + "".join(f"{stds[i]:<8.3f}" for i in (0, 2, 5, 8, 11)))

In [ ]:
he = propagate(lambda fan: (2 / fan) ** 0.5, relu)
xavier = propagate(lambda fan: (1 / fan) ** 0.5, tanh)
tiny = propagate(lambda fan: 0.01, tanh)

print(f"He     : layer 1 {he[0]:.3f}  ->  layer 12 {he[-1]:.3f}   (holds)")
print(f"Xavier : layer 1 {xavier[0]:.3f}  ->  layer 12 {xavier[-1]:.3f}   (decays)")
print(f"x0.01  : layer 1 {tiny[0]:.3f}  ->  layer  6 {tiny[5]:.3f}   (dead)")

assert he[-1] > 0.8            # He holds its signal
assert xavier[-1] < 0.3        # Xavier-on-tanh decays over 12 layers
assert tiny[5] < 1e-3          # x0.01 is gone

> **Read the Xavier row honestly.** It decays from 0.63 to 0.21 over twelve
> layers — much better than ×0.01, but not flat. Xavier is derived for the
> *linear* regime, and `tanh` shrinks its input a little at every layer. On the
> depths Xavier was designed for this is fine; at twelve layers you can see the
> drift. That drift is part of why ReLU plus He took over for deep stacks.

## Part C — Where He's factor of 2 comes from

It is not a tuning constant. ReLU sets about half of its inputs to zero, so it
destroys about half the variance. The 2 puts it back.

In [ ]:
rng = np.random.default_rng(3)
z = rng.standard_normal(500_000)          # zero-mean pre-activations

print(f"Var(z)         = {z.var():.4f}")
print(f"Var(relu(z))   = {relu(z).var():.4f}")
print(f"fraction zeroed = {(z <= 0).mean():.3f}")
print(f"\nratio Var(relu(z)) / Var(z) = {relu(z).var() / z.var():.4f}")
print("\nReLU keeps roughly half the variance, so He multiplies by 2 to")
print("compensate. Derived from the activation function, not tuned.")

In [ ]:
# Check it end to end: He on ReLU should hold, and 1/fan_in on ReLU should decay.
he_relu   = propagate(lambda fan: (2 / fan) ** 0.5, relu)
xav_relu  = propagate(lambda fan: (1 / fan) ** 0.5, relu)

print(f"He   on ReLU (factor 2)    : {he_relu[0]:.3f} -> {he_relu[-1]:.3f}")
print(f"1/fan on ReLU (no factor 2): {xav_relu[0]:.3f} -> {xav_relu[-1]:.3f}")
print("\nWithout the 2, the signal halves in variance at every layer.")
assert xav_relu[-1] < he_relu[-1]

## Part D — What Keras actually does by default

Worth knowing, because it is not what you would guess.

In [ ]:
try:
    import tensorflow as tf
    layer = tf.keras.layers.Dense(64, activation="relu")
    print("default kernel_initializer :", layer.kernel_initializer.__class__.__name__)
    print("default bias_initializer   :", layer.bias_initializer.__class__.__name__)
    print("\nglorot_uniform is XAVIER - even though the activation is ReLU.")
    print("For a deep ReLU stack, pass kernel_initializer='he_normal'.")
except ImportError:
    print("tensorflow not installed - skipping.")
    print("The point stands: Keras Dense defaults to glorot_uniform (Xavier)")
    print("and zeros for the bias, whatever activation you pass it.")

## What to take away

- **Make the weight variance depend on fan_in.** `Var(z) = fan_in × Var(w) ×
  Var(x)`, so `Var(w) = 1/fan_in` keeps the spread steady.
- **It self-adjusts:** wide layer → smaller weights, narrow layer → larger. You
  never pick a number.
- **Xavier/Glorot** is `2/(fan_in + fan_out)` — for **tanh and sigmoid**.
- **He** is `2/fan_in` — for **ReLU**. The 2 compensates for ReLU zeroing about
  half its inputs; it is **derived, not tuned**.
- Measured over 12 layers: **He holds std at ≈0.83 → 0.98**; Xavier decays
  0.63 → 0.21; ×0.01 is dead by layer 6.
- **Keras defaults to `glorot_uniform` even on ReLU layers.**
- **Normal vs uniform barely matters** — same variance, different shape.
- **Biases start at zero** — the random weights already broke the symmetry.

## Exercises

1. Xavier uses `2/(fan_in + fan_out)` rather than `1/fan_in`. Build a network
   with strongly *unequal* widths (say 512 → 32 → 512) and measure the forward
   and backward variance under both. What is the trade Xavier is making?
2. Swap `he_normal` for `he_uniform` (same variance, uniform shape) and rerun
   Part B. How much does the shape actually matter?
3. Push `DEPTH` to 50. Does He still hold? At what depth does it start to drift,
   and what would you reach for then?
4. He is derived for ReLU. Derive the equivalent factor for **leaky ReLU** with
   slope `α`, then verify it numerically the way Part C did.
5. Set biases to a small positive constant (028's dying-ReLU trick) on top of He.
   Does it change the activation std profile, or only the dead-unit count?